# 01 · Exploratory Data Analysis — iNaturalist (Insecta)

Professional EDA over the cleaned, 70/15/15-split dataset produced by
`00_data_ready.ipynb`. Covers dataset overview, image statistics, quality
analysis, annotation consistency, and visualizations. Heavy logic is imported
from `src/data_pipeline/eda.py`; figures are written to `data/interim/eda/`.

Methodology & sources: `docs/eda_best_practices.md`.

In [1]:
import sys, os, json, random
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")                       # headless
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

here = Path.cwd()
for cand in [here, *here.parents]:
    if (cand / "src" / "config.py").exists():
        os.chdir(cand); sys.path.insert(0, str(cand)); break
REPO = Path.cwd()
from src import config as C
from src.data_pipeline import eda

EDA = C.EDA_DIR; EDA.mkdir(parents=True, exist_ok=True)
df = eda.load_manifest_df()
paths = df["path"].tolist()
SUMMARY = {}
def save(fig, name):
    p = EDA / name; fig.tight_layout(); fig.savefig(p, dpi=110, bbox_inches="tight"); plt.close(fig)
    print("saved", p); return str(p)
print("images:", len(df), "| classes:", df['class_id'].nunique())

images: 151545 | classes: 2526


## 1 · Dataset overview
Image counts, class counts, split distribution, dataset size on disk.

In [2]:
sizes = df.assign(bytes=[ (REPO/p).stat().st_size for p in paths ])
overview = {
    "images_total": int(len(df)),
    "num_classes": int(df["class_id"].nunique()),
    "num_orders": int(df["order"].nunique()),
    "num_families": int(df["family"].nunique()),
    "bee_images": int(df["is_bee"].sum()),
    "split_counts": df["split"].value_counts().to_dict(),
    "split_fractions": {k: round(v/len(df),4) for k,v in df["split"].value_counts().items()},
    "disk_size_gb": round(sizes["bytes"].sum()/1e9, 2),
    "mean_image_kb": round(sizes["bytes"].mean()/1e3, 1),
}
SUMMARY["overview"] = overview
print(json.dumps(overview, indent=2))

{
  "images_total": 151545,
  "num_classes": 2526,
  "num_orders": 17,
  "num_families": 190,
  "bee_images": 3720,
  "split_counts": {
    "train": 106077,
    "test": 22734,
    "val": 22734
  },
  "split_fractions": {
    "train": 0.7,
    "test": 0.15,
    "val": 0.15
  },
  "disk_size_gb": 11.64,
  "mean_image_kb": 76.8
}


## 2 · Class & order distribution + imbalance

In [3]:
counts = eda.class_distribution(df)
orders = eda.order_distribution(df)
imb = eda.imbalance_metrics(counts)
SUMMARY["imbalance"] = imb
print("imbalance:", json.dumps(imb, indent=2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(counts.values, bins=range(int(counts.min()), int(counts.max())+2), color="#4C72B0", align="left")
ax[0].set(title="Per-class image counts", xlabel="images / class", ylabel="# classes")
orders.plot.barh(ax=ax[1], color="#55A868"); ax[1].invert_yaxis()
ax[1].set(title=f"Images per taxonomic order (n={len(orders)})", xlabel="images")
SUMMARY["fig_class_order"] = save(fig, "class_and_order_distribution.png")

imbalance: {
  "num_classes": 2526,
  "min": 59.0,
  "max": 60.0,
  "mean": 59.99406175771971,
  "imbalance_ratio": 1.0169491525423728,
  "gini": 0.0001
}


saved /home/manheim666/Desktop/Bee-A-Hero/data/interim/eda/class_and_order_distribution.png


## 3 · Split distribution, per-class coverage & bee subset

In [4]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sc = df["split"].value_counts()
ax[0].pie(sc.values, labels=[f"{k}\n{v:,}" for k,v in sc.items()], autopct="%1.1f%%",
          colors=["#4C72B0","#DD8452","#55A868"]); ax[0].set_title("Split distribution (70/15/15)")

bee = df.groupby(["split","is_bee"]).size().unstack(fill_value=0)
bee.columns = ["non-bee","bee"]
bee.plot.bar(ax=ax[1], stacked=True, color=["#C44E52","#8172B3"]); ax[1].set_title("Bee vs non-bee per split")
ax[1].set_xlabel("split"); ax[1].tick_params(axis="x", rotation=0)

order_split = df.pivot_table(index="order", columns="split", values="path", aggfunc="count", fill_value=0)
sns.heatmap(order_split, ax=ax[2], cmap="viridis", cbar_kws={"label":"images"})
ax[2].set_title("Order × split heatmap")
SUMMARY["fig_split"] = save(fig, "split_bee_order_heatmap.png")

# every class present in every split?
cov = (df.pivot_table(index="class_name", columns="split", values="path", aggfunc="count", fill_value=0) > 0)
SUMMARY["every_class_all_splits"] = bool(cov.all().all())
print("every class in all splits:", SUMMARY["every_class_all_splits"])

saved /home/manheim666/Desktop/Bee-A-Hero/data/interim/eda/split_bee_order_heatmap.png
every class in all splits: True


## 4 · Image statistics (sampled)
Resolution, aspect ratio, brightness, contrast, blur, colour mode. Sampled for speed (representative of 151k).

In [5]:
stats = eda.sample_image_stats(paths, sample=12000, workers=os.cpu_count())
stats = stats.dropna(subset=["width"])
res = {"width_min_med_max":[int(stats.width.min()),int(stats.width.median()),int(stats.width.max())],
       "height_min_med_max":[int(stats.height.min()),int(stats.height.median()),int(stats.height.max())],
       "aspect_med": round(float(stats.aspect.median()),3),
       "brightness_med": round(float(stats.brightness.median()),1),
       "modes": stats["mode"].value_counts().to_dict()}
SUMMARY["image_stats_sampled"] = res
print(json.dumps(res, indent=2))

fig, ax = plt.subplots(2, 3, figsize=(15, 8))
ax[0,0].hist(stats.width, bins=40, color="#4C72B0"); ax[0,0].set_title("Width (px)")
ax[0,1].hist(stats.height, bins=40, color="#4C72B0"); ax[0,1].set_title("Height (px)")
ax[0,2].scatter(stats.width, stats.height, s=4, alpha=.2, color="#4C72B0"); ax[0,2].set(title="Resolution scatter", xlabel="w", ylabel="h")
ax[1,0].boxplot(stats.aspect.dropna(), vert=True); ax[1,0].set_title("Aspect ratio (w/h)")
ax[1,1].hist(stats.brightness, bins=40, color="#DD8452"); ax[1,1].set_title("Brightness (mean gray)")
ax[1,2].hist(np.log1p(stats.blur), bins=40, color="#55A868"); ax[1,2].set_title("Blur: log(1+var(Laplacian))")
SUMMARY["fig_imgstats"] = save(fig, "image_statistics.png")

{
  "width_min_med_max": [
    161,
    500,
    500
  ],
  "height_min_med_max": [
    165,
    379,
    500
  ],
  "aspect_med": 1.305,
  "brightness_med": 124.9,
  "modes": {
    "RGB": 12000
  }
}


/tmp/ipykernel_29093/287792097.py:15: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  ax[1,0].boxplot(stats.aspect.dropna(), vert=True); ax[1,0].set_title("Aspect ratio (w/h)")


saved /home/manheim666/Desktop/Bee-A-Hero/data/interim/eda/image_statistics.png


## 5 · Quality analysis
Grayscale, blank, blurry, exposure outliers, near-duplicates. (Corrupted = 0, verified in `00_data_ready`.)

In [6]:
blur_thr = float(np.percentile(stats.blur, 5))     # bottom 5% as "blurry" flag
quality = {
    "grayscale_images_sampled": int(stats.is_gray.sum()),
    "blank_images_sampled": int(stats.blank.sum()),
    "blurry_flag_sampled(<=p5)": int((stats.blur <= blur_thr).sum()),
    "very_dark_sampled(bright<30)": int((stats.brightness < 30).sum()),
    "very_bright_sampled(bright>225)": int((stats.brightness > 225).sum()),
    "sample_size": int(len(stats)),
}
near = eda.find_near_duplicates(paths, sample=8000)
quality["near_duplicate_groups_sampled"] = len(near)
SUMMARY["quality"] = quality
print(json.dumps(quality, indent=2))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
stats["mode"].value_counts().plot.bar(ax=ax[0], color="#8172B3"); ax[0].set_title("Colour mode"); ax[0].tick_params(axis="x", rotation=0)
ax[1].hist(stats.contrast, bins=40, color="#C44E52"); ax[1].set_title("Contrast (gray std)")
SUMMARY["fig_quality"] = save(fig, "quality_mode_contrast.png")

{
  "grayscale_images_sampled": 25,
  "blank_images_sampled": 0,
  "blurry_flag_sampled(<=p5)": 600,
  "very_dark_sampled(bright<30)": 46,
  "very_bright_sampled(bright>225)": 29,
  "sample_size": 12000,
  "near_duplicate_groups_sampled": 0
}
saved /home/manheim666/Desktop/Bee-A-Hero/data/interim/eda/quality_mode_contrast.png


## 6 · Sample images
Deterministic random examples across splits.

In [7]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
grid = eda.sample_grid_paths(df, per_split=8)
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for axx, p in zip(axes.ravel(), grid):
    try:
        axx.imshow(Image.open(REPO/p).convert("RGB"));
    except Exception: pass
    axx.axis("off")
    axx.set_title(Path(p).parts[-2].split("_")[-1][:12], fontsize=7)
fig.suptitle("Random samples — rows: train / val / test", y=1.02)
SUMMARY["fig_samples"] = save(fig, "sample_grid.png")

saved /home/manheim666/Desktop/Bee-A-Hero/data/interim/eda/sample_grid.png


## 7 · EDA summary

In [8]:
(EDA / "eda_summary.json").write_text(json.dumps(SUMMARY, indent=2))
print(json.dumps({k:v for k,v in SUMMARY.items() if not str(k).startswith("fig_")}, indent=2))
print("\nfigures in", EDA)

{
  "overview": {
    "images_total": 151545,
    "num_classes": 2526,
    "num_orders": 17,
    "num_families": 190,
    "bee_images": 3720,
    "split_counts": {
      "train": 106077,
      "test": 22734,
      "val": 22734
    },
    "split_fractions": {
      "train": 0.7,
      "test": 0.15,
      "val": 0.15
    },
    "disk_size_gb": 11.64,
    "mean_image_kb": 76.8
  },
  "imbalance": {
    "num_classes": 2526,
    "min": 59.0,
    "max": 60.0,
    "mean": 59.99406175771971,
    "imbalance_ratio": 1.0169491525423728,
    "gini": 0.0001
  },
  "every_class_all_splits": true,
  "image_stats_sampled": {
    "width_min_med_max": [
      161,
      500,
      500
    ],
    "height_min_med_max": [
      165,
      379,
      500
    ],
    "aspect_med": 1.305,
    "brightness_med": 124.9,
    "modes": {
      "RGB": 12000
    }
  },
  "quality": {
    "grayscale_images_sampled": 25,
    "blank_images_sampled": 0,
    "blurry_flag_sampled(<=p5)": 600,
    "very_dark_sampled(bright<3